In [7]:
!pip install gensim

import re
import numpy as np
import pandas as pd
import torch
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split

df = pd.read_csv("07-yelp-dataset.txt", sep="\t", names=["review", "label"])

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["review"] = df["review"].apply(preprocess_text)

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

X_train_tokens = [review.split() for review in X_train]
X_test_tokens = [review.split() for review in X_test]

w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=200,
    window=5,
    min_count=1,
    sg=1,
    seed=42
)

def get_review_vector(tokens, model, vector_size=200):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size, dtype=np.float32)
    return np.mean(vectors, axis=0)

X_train_w2v = np.array([get_review_vector(tokens, w2v_model) for tokens in X_train_tokens])
X_test_w2v = np.array([get_review_vector(tokens, w2v_model) for tokens in X_test_tokens])

X_train_tensor = torch.FloatTensor(X_train_w2v)
X_test_tensor = torch.FloatTensor(X_test_w2v)

y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

print("Training Word2Vec shape:", X_train_tensor.shape)
print("Testing Word2Vec shape:", X_test_tensor.shape)

Training Word2Vec shape: torch.Size([800, 200])
Testing Word2Vec shape: torch.Size([200, 200])


In [8]:
import torch.nn as nn
from sklearn.metrics import accuracy_score

input_size = X_train_tensor.shape[1]

model_w2v = nn.Sequential(
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

print(model_w2v)

loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_w2v.parameters(), lr=0.001)

epochs = 15

for epoch in range(epochs):
    model_w2v.train()
    outputs = model_w2v(X_train_tensor)
    loss = loss_function(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

model_w2v.eval()
with torch.no_grad():
    test_logits = model_w2v(X_test_tensor)
    test_probabilities = torch.sigmoid(test_logits)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

test_accuracy = accuracy_score(y_test, predictions)
print(f"Word2Vec Model Test Accuracy: {test_accuracy:.3f}")

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)
Epoch 5/15 - Loss: 0.6941
Epoch 10/15 - Loss: 0.6934
Epoch 15/15 - Loss: 0.6931
Word2Vec Model Test Accuracy: 0.480
